# SHAP Analysis: Explaining XGBoost

**Chapter 5: Advanced Classification Methods**

## Learning Objectives

- Understand SHAP (SHapley Additive exPlanations)
- Install and use the SHAP library
- Create SHAP summary plots
- Interpret individual predictions
- Use SHAP for model debugging and insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import shap

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"SHAP version: {shap.__version__}")

## The Problem: Black Box Models

XGBoost is highly accurate, but it's a **black box**:
- We know it works well
- But we don't know **why** it makes specific predictions
- Hard to trust or debug

**SHAP solves this** by explaining the contribution of each feature to every prediction.

## What is SHAP?

**SHAP (SHapley Additive exPlanations)** uses game theory to assign each feature an importance value for a particular prediction.

**Key Idea:** How much does each feature contribute to moving the prediction from the base value (average prediction) to the final prediction?

**Example:**
- Base xG (average): 0.10
- Predicted xG for a specific shot: 0.65
- SHAP tells us: distance contributed +0.30, angle contributed +0.20, etc.

## Load and Prepare Data

We'll use the same xG dataset from the previous notebook.

In [ ]:
# Load and prepare data (same as Notebook 4)
matches = sb.matches(competition_id=16, season_id=4)
events = sb.events(match_id=22912)
shots = events[events['type'] == 'Shot'].copy()

# Feature engineering
def extract_shot_features(shots_df):
    df = shots_df.copy()
    df['distance_to_goal'] = df['location'].apply(
        lambda loc: np.sqrt((120 - loc[0])**2 + (40 - loc[1])**2) if isinstance(loc, list) else None
    )
    def calculate_angle(loc):
        if not isinstance(loc, list):
            return None
        x, y = loc[0], loc[1]
        return np.abs(np.arctan((40 - y) / (120 - x))) * 180 / np.pi
    df['angle_to_goal'] = df['location'].apply(calculate_angle)
    df['right_foot'] = (df['shot_body_part'] == 'Right Foot').astype(int)
    df['open_play'] = (df['shot_type'] == 'Open Play').astype(int)
    df['goal'] = (df['shot_outcome'] == 'Goal').astype(int)
    return df

shots = extract_shot_features(shots)

feature_cols = ['distance_to_goal', 'angle_to_goal', 'right_foot', 'open_play']
X = shots[feature_cols].dropna()
y = shots.loc[X.index, 'goal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data prepared: {len(X_train)} training, {len(X_test)} test shots")

## Train XGBoost Model

In [ ]:
# Train XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    objective='binary:logistic',
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)

print(f"Model trained successfully")
print(f"Test accuracy: {xgb_model.score(X_test_scaled, y_test):.3f}")

## Create SHAP Explainer

In [ ]:
# Create SHAP explainer for tree-based models
explainer = shap.TreeExplainer(xgb_model)

# Calculate SHAP values for test set
shap_values = explainer.shap_values(X_test_scaled)

print(f"SHAP values calculated")
print(f"Shape: {shap_values.shape}")
print(f"\nBase value (average prediction): {explainer.expected_value:.3f}")

## SHAP Summary Plot

This shows the impact of each feature across all predictions.

In [ ]:
# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_cols, show=False)
plt.title('SHAP Summary Plot: Feature Impact on xG Prediction', fontsize=14)
plt.tight_layout()
plt.show()

print("\nHow to read this plot:")
print("- Features ranked by importance (top to bottom)")
print("- Each dot is a shot")
print("- Red = high feature value, Blue = low feature value")
print("- Right = increases xG, Left = decreases xG")

## Interpreting the Summary Plot

**Example Insights:**

1. **distance_to_goal** (most important)
   - Red dots (far shots) on the left → decrease xG
   - Blue dots (close shots) on the right → increase xG
   - **Makes sense!** Closer shots are more likely to score

2. **angle_to_goal**
   - Red dots (wide angle) on the right → increase xG
   - Blue dots (narrow angle) on the left → decrease xG
   - **Makes sense!** Better angles increase scoring chance

This validates that the model learned sensible soccer patterns!

## SHAP Force Plot: Individual Prediction

Let's explain a single prediction in detail.

In [ ]:
# Select a specific shot to explain
shot_idx = 0
shot_features = X_test_scaled[shot_idx]
shot_prediction = xgb_model.predict_proba(shot_features.reshape(1, -1))[0, 1]

print(f"Shot #{shot_idx}:")
for i, feat in enumerate(feature_cols):
    print(f"  {feat}: {shot_features[i]:.2f}")
print(f"\nPredicted xG: {shot_prediction:.3f}")

# Force plot (shows how features push prediction from base value)
shap.force_plot(
    explainer.expected_value,
    shap_values[shot_idx],
    X_test_scaled[shot_idx],
    feature_names=feature_cols,
    matplotlib=True,
    show=False
)
plt.title(f'SHAP Force Plot for Shot #{shot_idx}', fontsize=14)
plt.tight_layout()
plt.show()

## SHAP Dependence Plot

Shows how a single feature affects predictions across its range.

In [ ]:
# Dependence plot for distance_to_goal
plt.figure(figsize=(10, 6))
shap.dependence_plot(
    'distance_to_goal',
    shap_values,
    X_test_scaled,
    feature_names=feature_cols,
    show=False
)
plt.title('SHAP Dependence Plot: Distance to Goal', fontsize=14)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- X-axis: Feature value (distance)")
print("- Y-axis: SHAP value (impact on prediction)")
print("- As distance increases, SHAP value decreases (negative impact on xG)")

## Using SHAP for Model Debugging

SHAP can reveal if your model learned the wrong patterns.

In [ ]:
# Check if any features have unexpected relationships
for i, feat in enumerate(feature_cols):
    shap_mean = np.mean(shap_values[:, i])
    print(f"{feat}: Average SHAP value = {shap_mean:.4f}")

print("\nExpected patterns:")
print("✓ distance_to_goal: Negative (closer = higher xG)")
print("✓ angle_to_goal: Positive (wider angle = higher xG)")
print("\nIf patterns don't match soccer intuition, investigate:") 
print("- Feature engineering errors")
print("- Data quality issues")
print("- Model overfitting")

## Summary

In this notebook, we:

1. ✅ Understood SHAP and why it's needed
2. ✅ Created SHAP explainer for XGBoost
3. ✅ Generated summary plots showing global feature importance
4. ✅ Explained individual predictions with force plots
5. ✅ Used dependence plots to understand feature relationships
6. ✅ Learned how to use SHAP for model debugging

## Key Takeaways

- SHAP makes **black box models interpretable**
- **Summary plots** show global feature importance
- **Force plots** explain individual predictions
- **Dependence plots** reveal feature-prediction relationships
- Use SHAP to **validate** that models learn sensible patterns

## Next Steps

Now that we understand how to build and interpret tree-based models, let's explore **hyperparameter tuning** for XGBoost in the extras!

## Practice Exercises

1. **Waterfall Plot**: Use shap.waterfall_plot() for another visualization
2. **Feature Interactions**: Use shap.dependence_plot() with interaction_index
3. **Different Models**: Compare SHAP values between Random Forest and XGBoost
4. **Real-World Application**: Explain predictions to a coach or analyst
5. **SHAP for Regression**: Apply to a regression problem (e.g., player market value)